In [1]:
import re
from typing import List, Optional

from pypdf import PdfReader

In [2]:
def extract_text_from_pdf(
    file_path: str, page_numbers: Optional[List[int]] = None
) -> str:
    """
    Extract plain text from specified pages of a PDF file.

    Args:
        file_path: Path to the PDF file
        page_numbers: List of page numbers to extract (1-indexed, like in the PDF).
                      If None, extracts all pages.

    Returns:
        String containing extracted text from the specified pages

    Raises:
        FileNotFoundError: If the PDF file doesn't exist
        ValueError: If page numbers are invalid
    """
    try:
        # Open and read the PDF
        reader = PdfReader(file_path)
        total_pages = len(reader.pages)

        # Determine which pages to extract
        if page_numbers is None:
            pages_to_extract = range(1, total_pages + 1)
        else:
            # Validate page numbers (convert to 0-indexed internally)
            pages_to_extract = []
            for page_num in page_numbers:
                if page_num < 1 or page_num > total_pages:
                    raise ValueError(
                        f"Page number {page_num} is out of range. PDF has {total_pages} page(s)."
                    )
                pages_to_extract.append(page_num)

        # Extract text from each page
        extracted_text = []
        for page_num in pages_to_extract:
            # Convert to 0-indexed for pypdf
            page = reader.pages[page_num - 1]
            text = page.extract_text()
            if text:  # Only add non-empty text
                extracted_text.append(text)

        # Join all extracted text with page separators
        return "\n\n".join(extracted_text)

    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {file_path}")
    except Exception as e:
        raise Exception(f"Error extracting text from PDF: {str(e)}")


In [3]:
text = extract_text_from_pdf(
    "Palmares-Pavillon-Bleu-2025.pdf",
    page_numbers=[2, 3, 4, 5],
)
print(text)

2PAVILLON BLEU  
SAISON 2025
Retrouvez cet été toutes ces plages et leur drapeau Pavillon Bleu !
Palmarès plages
388 plages labellisées dans 182 communes
AUVERGNE-RHÔNE-ALPES
22 
03. ALLIER | 1  | Vieure Plage du plan d’eau 
de Vieure 
07. ARDÈCHE  | 2  |  Saint-Martin-
d’Ardèche Plage du Grain de sel | Saint-Just-
d’Ardèche Plage de Saint-Just-d’Ardèche
15. CANTAL | 1  | Trémouille Lastioulles
43. HAUTE-LOIRE | 2  | Champagnac-le-
Vieux Plage du Plan d’eau  | Le Bouchet-
Saint-Nicolas Plage du lac du Bouchet 
74. HAUTE-SAVOIE | 9  | Evian-les-Bains 
Plage du Centre Nautique Evian | Neuvecelle 
Grande Rive | Publier Plage municipale 
Amphion-les-bains | Saint-Gingolph Plage 
municipale | Saint-Jorioz Plage municipale 
| Sciez-sur-Leman Sciez Municipale | 
Thonon-les-Bains Plage de Saint-Disdille, 
Plage du Centre Nautique | Veyrier-du-Lac 
Plage La Brune
63. PUY-DE-DÔME  | 6  |  Aubusson 
d’Auvergne Plage du lac d’Aubusson 
d’Auvergne | Aydat  Plage d’Aydat | 
Chambon-sur-Lac Plage de 

In [4]:
regions = [
    "AUVERGNE-RHÔNE-ALPES",
    "BOURGOGNE-FRANCHE-COMTÉ",
    "BRETAGNE",
    "CENTRE-VAL DE LOIRE",
    "CORSE",
    "GRAND EST",
    "HAUTS-DE-FRANCE",
    "ÎLE-DE-FRANCE",
    "NORMANDIE",
    "NOUVELLE-AQUITAINE",
    "OCCITANIE",
    "OUTRE-MER",
    "PAYS DE LA LOIRE",
    "PROVENCE-ALPES-CÔTE D'AZUR",
]

In [5]:
departements = [
    "01. AIN",
    "03. ALLIER",
    "07. ARDÈCHE",
    "15. CANTAL",
    "43. HAUTE-LOIRE",
    "74. HAUTE-SAVOIE",
    "63. PUY-DE-DÔME",
    "73. SAVOIE",
    "21. CÔTE-D'OR",
    "25. DOUBS",
    "39. JURA",
    "58. NIÈVRE",
    "29. FINISTÈRE",
    "35. ILLE-ET-VILAINE",
    "56. MORBIHAN",
    "36. INDRE",
    "45. LOIRET",
    "2A. CORSE-DU-SUD",
    "08. ARDENNES",
    "59. NORD",
    "80. SOMME",
    "77. SEINE-ET-MARNE",
    "14. CALVADOS",
    "50. MANCHE",
    "76. SEINE-MARITIME",
    "17. CHARENTE-MARITIME",
    "19. CORRÈZE",
    "24. DORDOGNE",
    "33. GIRONDE",
    "87. HAUTE-VIENNE",
    "40. LANDES",
    "64. PYRÉNÉES-ATLANTIQUES",
    "11. AUDE",
    "12. AVEYRON",
    "30. GARD",
    "31. HAUTE-GARONNE",
    "34. HÉRAULT",
    "46. LOT",
    "48. LOZÈRE",
    "66. PYRÉNÉES-ORIENTALES",
    "81. TARN",
    "82. TARN-ET-GARONNE",
    "97. LA RÉUNION",
    "98. POLYNÉSIE FRANÇAISE",
    "44. LOIRE-ATLANTIQUE",
    "72. SARTHE",
    "85. VENDÉE",
    "04. ALPES-DE-HAUTE-PROVENCE",
    "06. ALPES-MARITIMES",
    "13. BOUCHES-DU-RHÔNE",
    "05. HAUTES-ALPES",
]

In [6]:
trim_strings = (
    "Retrouvez cet été toutes ces plages et leur drapeau Pavillon Bleu !",
    "Palmarès plages",
    "388 plages labellisées dans 182 communes",
    "Palmarès plages (suite)",
    "2PAVILLON BLEU",
    "3PAVILLON BLEU",
    "SAISON 2025",
    "Légende : retrouvez en italique et en bleu clair les plages labellisées pour la première fois.",
)

In [7]:
for unwanted in trim_strings:
    text = text.replace(unwanted, "")
text = "".join(text.split("\n")).strip()

In [8]:
# We need to change the name of the only departments that have the region's name in it...
text = text.replace("CORSE-DU-", "COXXXRSE-DU-")

In [9]:
print(text)

AUVERGNE-RHÔNE-ALPES22 03. ALLIER | 1  | Vieure Plage du plan d’eau de Vieure 07. ARDÈCHE  | 2  |  Saint-Martin-d’Ardèche Plage du Grain de sel | Saint-Just-d’Ardèche Plage de Saint-Just-d’Ardèche15. CANTAL | 1  | Trémouille Lastioulles43. HAUTE-LOIRE | 2  | Champagnac-le-Vieux Plage du Plan d’eau  | Le Bouchet-Saint-Nicolas Plage du lac du Bouchet 74. HAUTE-SAVOIE | 9  | Evian-les-Bains Plage du Centre Nautique Evian | Neuvecelle Grande Rive | Publier Plage municipale Amphion-les-bains | Saint-Gingolph Plage municipale | Saint-Jorioz Plage municipale | Sciez-sur-Leman Sciez Municipale | Thonon-les-Bains Plage de Saint-Disdille, Plage du Centre Nautique | Veyrier-du-Lac Plage La Brune63. PUY-DE-DÔME  | 6  |  Aubusson d’Auvergne Plage du lac d’Aubusson d’Auvergne | Aydat  Plage d’Aydat | Chambon-sur-Lac Plage de Chambon |  Murol Grande plage du lac Chambon de Murol | Saint-Rémy-Sur-Durolle Plan d’eau des Prades | Thiers Plan d’eau d’Iloa73. SAVOIE | 1  | Grésy-sur-Isère Base de loisirs 

In [10]:
def split_by_pattern_list(pattern_list, text):
    pattern = "|".join(re.escape(item) for item in pattern_list)
    # Add parentheses to capture the delimiters
    pattern_with_capture = f"({pattern})"
    return re.split(pattern_with_capture, text)

In [11]:
# Split the text while keeping the regions
splitted = split_by_pattern_list(regions, text)

In [12]:
splitted = splitted[1:]
splitted

['AUVERGNE-RHÔNE-ALPES',
 '22 03. ALLIER | 1  | Vieure Plage du plan d’eau de Vieure 07. ARDÈCHE  | 2  |  Saint-Martin-d’Ardèche Plage du Grain de sel | Saint-Just-d’Ardèche Plage de Saint-Just-d’Ardèche15. CANTAL | 1  | Trémouille Lastioulles43. HAUTE-LOIRE | 2  | Champagnac-le-Vieux Plage du Plan d’eau  | Le Bouchet-Saint-Nicolas Plage du lac du Bouchet 74. HAUTE-SAVOIE | 9  | Evian-les-Bains Plage du Centre Nautique Evian | Neuvecelle Grande Rive | Publier Plage municipale Amphion-les-bains | Saint-Gingolph Plage municipale | Saint-Jorioz Plage municipale | Sciez-sur-Leman Sciez Municipale | Thonon-les-Bains Plage de Saint-Disdille, Plage du Centre Nautique | Veyrier-du-Lac Plage La Brune63. PUY-DE-DÔME  | 6  |  Aubusson d’Auvergne Plage du lac d’Aubusson d’Auvergne | Aydat  Plage d’Aydat | Chambon-sur-Lac Plage de Chambon |  Murol Grande plage du lac Chambon de Murol | Saint-Rémy-Sur-Durolle Plan d’eau des Prades | Thiers Plan d’eau d’Iloa73. SAVOIE | 1  | Grésy-sur-Isère Base de l

In [13]:
import csv
import unicodedata
import re as re_module

YEAR = 2025
OUTPUT_FILE = "terragir_officiel_2025.csv"

# Normalize all comparison patterns to NFC for robust matching
regions_nfc = [unicodedata.normalize("NFC", r) for r in regions]
departements_nfc = [unicodedata.normalize("NFC", d) for d in departements]

# Words that signal the start of a beach name
BEACH_KEYWORDS = [
    "plage", "base", "plan", "lac", "étang", "etang",
    "baignade", "complexe", "camping", "piscine", "grande", "grand’",
]


def extract_commune_beaches(town_entry):
    items = [i.strip() for i in town_entry.split(",")]
    if not items or not items[0]:
        return None, []

    first = items[0]
    words = first.split()

    # For "Le/La/Les"-prefixed towns, skip first 2 words for keyword search
    # For others, skip first 1 word
    starts_with_article = words[0].lower().strip("’") in ("le", "la", "les")
    skip = 2 if starts_with_article else 1

    split_idx = None
    for i in range(skip, len(words)):
        w = words[i].lower().replace("’", "")
        for kw in BEACH_KEYWORDS:
            if w.startswith(kw) or w == kw:
                split_idx = i
                break
        if split_idx is not None:
            break

    if split_idx is not None:
        commune = " ".join(words[:split_idx])
        first_beach = " ".join(words[split_idx:])
    else:
        if starts_with_article and len(words) >= 2:
            # Special case: "La Grande Motte" is 3 words
            if len(words) >= 4 and words[1] == "Grande" and words[2].startswith("Motte"):
                commune = " ".join(words[:3])
                first_beach = " ".join(words[3:])
            else:
                commune = " ".join(words[:2])
                first_beach = " ".join(words[2:])
        else:
            commune = words[0]
            first_beach = " ".join(words[1:])

    beaches = []
    if first_beach:
        beaches.append(first_beach)
    beaches.extend(items[1:])
    beaches = [b.strip().rstrip(".") for b in beaches if b.strip()]

    return commune, beaches


def dept_number_and_name(dept_code):
    if ". " in dept_code:
        num, name = dept_code.split(". ", 1)
        return num.strip(), name.strip()
    return "", dept_code


# Department-to-region override (for cases where Unicode normalization in PDF
# prevents clean region splitting)
DEPT_TO_REGION = {
    "04. ALPES-DE-HAUTE-PROVENCE": "PROVENCE-ALPES-CÔTE D'AZUR",
    "06. ALPES-MARITIMES": "PROVENCE-ALPES-CÔTE D'AZUR",
    "13. BOUCHES-DU-RHÔNE": "PROVENCE-ALPES-CÔTE D'AZUR",
    "05. HAUTES-ALPES": "PROVENCE-ALPES-CÔTE D'AZUR",
}

records = []

for i in range(1, len(splitted), 2):
    region = splitted[i - 1].strip()
    region = unicodedata.normalize("NFC", region)
    region_content = splitted[i].strip()
    region_content = unicodedata.normalize("NFC", region_content)

    region_content = region_content.replace("COXXXRSE-DU-", "CORSE-DU-")
    region_content = region_content.replace("PALMARÈS 2025 DES LABELLISÉS PAVILLON BLEU", "")

    # Collapse multiple spaces for robust department matching
    region_content = re_module.sub(r"\s+", " ", region_content)

    splitted_dpt = split_by_pattern_list(departements_nfc, region_content)
    splitted_dpt = splitted_dpt[1:]

    current_dept = None
    for item in splitted_dpt:
        item = item.strip()
        if not item:
            continue

        if item in departements_nfc:
            current_dept = item
            # Override region if department belongs elsewhere
            if current_dept in DEPT_TO_REGION:
                region = DEPT_TO_REGION[current_dept]
            continue

        if current_dept is None:
            continue

        # Split department content by | to get: count | town1 | town2 | ...
        parts = [p.strip() for p in item.split("|") if p.strip()]

        # First part is the beach count number
        if parts and parts[0].isdigit():
            parts = parts[1:]

        if not parts:
            continue

        dept_num, dept_name = dept_number_and_name(current_dept)

        for town_raw in parts:
            if town_raw.isdigit() or "(suite)" in town_raw.lower():
                continue

            commune, beaches = extract_commune_beaches(town_raw)
            if commune and beaches and any(c.isalpha() for c in commune):
                records.append({
                    "year": YEAR,
                    "commune": commune.upper(),
                    "department_name": dept_name,
                    "department_number": dept_num,
                    "region": region,
                    "beach_flag": 1,
                    "port_flag": 0,
                    "places": ", ".join(beaches),
                })

# Write CSV
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    fields = [
        "year", "commune", "department_name", "department_number",
        "region", "beach_flag",
        "port_flag", "places",
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

print(f"Done! Wrote {len(records)} records to {OUTPUT_FILE}")
print("Known edge case: 'La Grande Motte' will appear as 'LA GRANDE' instead of 'LA GRANDE MOTTE' \u2014 fix manually in the CSV")


Done! Wrote 167 records to terragir_officiel_2025.csv
Known edge case: 'La Grande Motte' will appear as 'LA GRANDE' instead of 'LA GRANDE MOTTE' — fix manually in the CSV


In [14]:
# === CLEANUP: fix known PDF extraction artifacts ===
import re
import unicodedata

for rec in records:
    commune = rec["commune"]
    places = rec["places"]
    dept = rec["department_number"]

    # 1. Fix "PORT" -> "PORT-DE-BOUC" in Bouches-du-Rh\u00f4ne
    if commune == "PORT" and dept == "13":
        rec["commune"] = "PORT-DE-BOUC"
        rec["places"] = "Bottai"
        continue

    # 2. Fix La Baule-Escoublac (commune absorbed "Face \u00e0 l'Avenue de la ")
    if dept == "44" and "BAULE-ESCOUBLAC" in commune:
        rec["commune"] = "LA BAULE-ESCOUBLAC"
        parts = [p.strip() for p in rec["places"].split(",")]
        if parts and parts[0] == "Grande Dune":
            parts[0] = "Face \u00e0 l'Avenue de la Grande Dune"
        rec["places"] = ", ".join(parts)
        continue

    # 2b. Fix Bora Bora (one "Bora" captured as commune, second in places)
    if commune == "BORA" and dept == "98":
        rec["commune"] = "BORA BORA"
        parts = [p.strip() for p in rec["places"].split(",")]
        if parts and parts[0].startswith("Bora "):
            parts[0] = parts[0][5:]
        rec["places"] = ", ".join(parts)
        continue

    # 3. Fix commune deduplication
    words = commune.split()
    if len(words) >= 2:
        first_word = words[0]
        last_word = words[-1]

        if first_word == last_word:
            rec["commune"] = " ".join(words[:-1])
            places_parts = [p.strip() for p in rec["places"].split(",")]
            if places_parts:
                places_parts[0] = f"{last_word.capitalize()} {places_parts[0]}"
            rec["places"] = ", ".join(places_parts)

        elif "-" in first_word and last_word in first_word.split("-"):
            rec["commune"] = " ".join(words[:-1])
            places_parts = [p.strip() for p in rec["places"].split(",")]
            if places_parts:
                places_parts[0] = f"{last_word.capitalize()} {places_parts[0]}"
            rec["places"] = ", ".join(places_parts)

    # 4. Fix "NEUVIC D'USSEL LA" -> move trailing "LA" to beaches
    words = rec["commune"].split()
    if len(words) >= 2 and words[-1] in ("LA", "LE", "LES"):
        article = words[-1].title()
        rec["commune"] = " ".join(words[:-1])
        places_parts = [p.strip() for p in rec["places"].split(",")]
        if places_parts:
            places_parts[0] = f"{article} {places_parts[0]}"
        rec["places"] = ", ".join(places_parts)

# 5. Strip region-name artifacts from places
REGION_NAMES = [
    "PROVENCE-ALPES-C\u00d4TE D'AZUR", "AUVERGNE-RH\u00d4NE-ALPES",
    "BOURGOGNE-FRANCHE-COMT\u00c9", "BRETAGNE", "CENTRE-VAL DE LOIRE",
    "CORSE", "GRAND EST", "HAUTS-DE-FRANCE", "\u00ceLE-DE-FRANCE",
    "NORMANDIE", "NOUVELLE-AQUITAINE", "OCCITANIE", "OUTRE-MER",
    "PAYS DE LA LOIRE",
]
for rec in records:
    places = rec["places"]
    for name in REGION_NAMES:
        places = places.replace(name, "")
    places = re.sub(r"\s+\d{2}\b", "", places)
    places = places.strip().rstrip(",").strip()
    rec["places"] = places

# 6. Port artifact detection + reclassification
PORT_ARTIFACT_COMMUNES = {"PORT", "HALTE", "RELAIS"}
PORT_KEYWORDS = [
    "port de plaisance", "Port de plaisance", "Port de Plaisance",
    "halte fluviale", "Halte Fluviale",
    "Port fluvial", "Marina",
]

# Regex to remove embedded department codes like "67. BAS-RHIN", "2B. HAUTE-"
# (no \b anchor because codes often attach without word boundary)
dept_code_pattern = re.compile(r"\d{1,2}[A-Z]?\.\s*[A-Z\u00c9\u00c8\u00ca\u00cb\u00c0\u00c2\u00ce\u00cf\u00d4\u00db\u00dc\u00c7\u2019' -]+")

# Comprehensive department-to-region map for port records that inherit wrong regions
DEPT_REGION_MAP = {
    "01": "AUVERGNE-RH\u00d4NE-ALPES", "02": "HAUTS-DE-FRANCE",
    "03": "AUVERGNE-RH\u00d4NE-ALPES", "04": "PROVENCE-ALPES-C\u00d4TE D'AZUR",
    "05": "PROVENCE-ALPES-C\u00d4TE D'AZUR", "06": "PROVENCE-ALPES-C\u00d4TE D'AZUR",
    "07": "AUVERGNE-RH\u00d4NE-ALPES", "08": "GRAND EST",
    "09": "OCCITANIE", "10": "GRAND EST",
    "11": "OCCITANIE", "12": "OCCITANIE",
    "13": "PROVENCE-ALPES-C\u00d4TE D'AZUR", "14": "NORMANDIE",
    "15": "AUVERGNE-RH\u00d4NE-ALPES", "16": "NOUVELLE-AQUITAINE",
    "17": "NOUVELLE-AQUITAINE", "18": "CENTRE-VAL DE LOIRE",
    "19": "NOUVELLE-AQUITAINE", "2A": "CORSE",
    "2B": "CORSE", "21": "BOURGOGNE-FRANCHE-COMT\u00c9",
    "22": "BRETAGNE", "23": "NOUVELLE-AQUITAINE",
    "24": "NOUVELLE-AQUITAINE", "25": "BOURGOGNE-FRANCHE-COMT\u00c9",
    "26": "AUVERGNE-RH\u00d4NE-ALPES", "27": "NORMANDIE",
    "28": "CENTRE-VAL DE LOIRE", "29": "BRETAGNE",
    "30": "OCCITANIE", "31": "OCCITANIE",
    "32": "OCCITANIE", "33": "NOUVELLE-AQUITAINE",
    "34": "OCCITANIE", "35": "BRETAGNE",
    "36": "CENTRE-VAL DE LOIRE", "37": "CENTRE-VAL DE LOIRE",
    "38": "AUVERGNE-RH\u00d4NE-ALPES", "39": "BOURGOGNE-FRANCHE-COMT\u00c9",
    "40": "NOUVELLE-AQUITAINE", "41": "CENTRE-VAL DE LOIRE",
    "42": "AUVERGNE-RH\u00d4NE-ALPES", "43": "AUVERGNE-RH\u00d4NE-ALPES",
    "44": "PAYS DE LA LOIRE", "45": "CENTRE-VAL DE LOIRE",
    "46": "OCCITANIE", "47": "NOUVELLE-AQUITAINE",
    "48": "OCCITANIE", "49": "PAYS DE LA LOIRE",
    "50": "NORMANDIE", "51": "GRAND EST",
    "52": "GRAND EST", "53": "PAYS DE LA LOIRE",
    "54": "GRAND EST", "55": "GRAND EST",
    "56": "BRETAGNE", "57": "GRAND EST",
    "58": "BOURGOGNE-FRANCHE-COMT\u00c9", "59": "HAUTS-DE-FRANCE",
    "60": "HAUTS-DE-FRANCE", "61": "NORMANDIE",
    "62": "HAUTS-DE-FRANCE", "63": "AUVERGNE-RH\u00d4NE-ALPES",
    "64": "NOUVELLE-AQUITAINE", "65": "OCCITANIE",
    "66": "OCCITANIE", "67": "GRAND EST",
    "68": "GRAND EST", "69": "AUVERGNE-RH\u00d4NE-ALPES",
    "70": "BOURGOGNE-FRANCHE-COMT\u00c9", "71": "BOURGOGNE-FRANCHE-COMT\u00c9",
    "72": "PAYS DE LA LOIRE", "73": "AUVERGNE-RH\u00d4NE-ALPES",
    "74": "AUVERGNE-RH\u00d4NE-ALPES", "75": "\u00ceLE-DE-FRANCE",
    "76": "NORMANDIE", "77": "\u00ceLE-DE-FRANCE",
    "78": "\u00ceLE-DE-FRANCE", "79": "NOUVELLE-AQUITAINE",
    "80": "HAUTS-DE-FRANCE", "81": "OCCITANIE",
    "82": "OCCITANIE", "83": "PROVENCE-ALPES-C\u00d4TE D'AZUR",
    "84": "PROVENCE-ALPES-C\u00d4TE D'AZUR", "85": "PAYS DE LA LOIRE",
    "86": "NOUVELLE-AQUITAINE", "87": "NOUVELLE-AQUITAINE",
    "88": "GRAND EST", "89": "BOURGOGNE-FRANCHE-COMT\u00c9",
    "90": "BOURGOGNE-FRANCHE-COMT\u00c9", "91": "\u00ceLE-DE-FRANCE",
    "92": "\u00ceLE-DE-FRANCE", "93": "\u00ceLE-DE-FRANCE",
    "94": "\u00ceLE-DE-FRANCE", "95": "\u00ceLE-DE-FRANCE",
    "97": "OUTRE-MER", "98": "OUTRE-MER",
}

for rec in records:
    commune = rec["commune"]
    places = rec["places"]
    dept_num = rec["department_number"]
    is_port = False

    # Fix wrong region for port records based on department
    if dept_num in DEPT_REGION_MAP:
        correct_region = DEPT_REGION_MAP[dept_num]
        if rec["region"] != correct_region:
            rec["region"] = correct_region

    if commune in PORT_ARTIFACT_COMMUNES:
        is_port = True
        full_entry = f"{commune} {places}"

        # Find and remove embedded department codes (garbled merge artifacts)
        # Capture codes first so we can fix department numbers
        embed_codes = re.findall(r"(\d{1,2}[A-Z]?)\.\s*([A-Z\u00c9\u00c8\u00ca\u00cb\u00c0\u00c2\u00ce\u00cf\u00d4\u00db\u00dc\u00c7\u2019' -]+)", full_entry)
        full_entry = dept_code_pattern.sub("", full_entry)
        if embed_codes:
            num, name = embed_codes[0]
            num = num.strip()
            if num != rec["department_number"]:
                rec["department_number"] = num
                rec["department_name"] = name.strip().rstrip("-").strip()
                if num in DEPT_REGION_MAP:
                    rec["region"] = DEPT_REGION_MAP[num]

        # Normalize to NFC and uppercase for matching (PDF text may be NFD/mixed case)
        full_entry_nfc = unicodedata.normalize("NFC", full_entry)
        full_entry_haystack = full_entry_nfc.upper()
        # Remove trailing noise (string search avoids regex Unicode issues)
        for sep in ["CETTE ANN\u00c9E", "PALMAR\u00c8S", "L\u00c9GENDE"]:
            if sep in full_entry_haystack:
                idx = full_entry_haystack.index(sep)
                full_entry = full_entry_nfc[:idx]
                break
        # Remove known region names
        for name in REGION_NAMES:
            full_entry = full_entry.replace(name, "")
        # Clean up whitespace and trailing punctuation
        full_entry = re.sub(r"\s+", " ", full_entry).strip().rstrip(",").strip().rstrip(".").strip()
        # Strip trailing noise after a period (like "MEURTHE-ET-MOSELLE")
        full_entry = re.sub(r"\.[A-Z\u00c9\u00c8\u00ca\u00cb\u00c0\u00c2\u00ce\u00cf\u00d4\u00db\u00dc\u00c7\u2019\u0027 -]+$", "", full_entry)
        full_entry = full_entry.strip().rstrip(".").strip()

        if full_entry:
            # Extract town name from port entry (remove port prefix for commune)
            _town = full_entry.upper().strip()
            _port_prefixes = [
                "PORT DE PLAISANCE DE ", "PORT DE PLAISANCE ",
                "PORT FLUVIAL DES ", "PORT FLUVIAL DE ", "PORT FLUVIAL ",
                "PORT DE ", "PORT DES ", "PORT ",
                "HALTE FLUVIALE DE ", "HALTE FLUVIALE ",
                "HALTE DE ", "HALTE ",
                "RELAIS NAUTIQUE DE L'", "RELAIS NAUTIQUE DE ",
                "RELAIS NAUTIQUE D'", "RELAIS NAUTIQUE ",
            ]
            for _pfx in sorted(_port_prefixes, key=len, reverse=True):
                if _town.startswith(_pfx) and _town != _pfx:
                    _town = _town[len(_pfx):]
                    break
            _town = re.sub(r"\s*\(.*?\)", "", _town).strip()
            for _prep in ["DE LA ", "DE L'", "DU ", "DES ", "DE ", "D'"]:
                if _town.startswith(_prep):
                    _town = _town[len(_prep):]
                    break
            # Commune refinement: extract actual town from port names
            if _town:
                for _sep in [" DE ", " DES "]:
                    if _sep in _town:
                        _town = _town.rsplit(_sep, 1)[-1].strip()
                        break
                _person_port_map = {
                    "AJACCIO TINO ROSSI": "AJACCIO",
                    "CHARLES ORNANO": "AJACCIO",
                    "JEAN-BAPTISTE TOMI": "BASTIA",
                    "MICHEL ROTH": "LANEUVILLE-SUR-MEUSE",
                }
                if _town in _person_port_map:
                    _town = _person_port_map[_town]
                elif _town == "FRANCE":
                    _town = "NANCY"

            rec["commune"] = _town if _town else full_entry.upper()
            rec["places"] = full_entry

    elif any(kw.lower() in places.lower() for kw in PORT_KEYWORDS):
        is_port = True

    if is_port:
        rec["port_flag"] = 1
        rec["beach_flag"] = 0

# 6b. Manual department corrections for port entries where PDF layout
#     causes the wrong department to be inherited
PORT_DEPT_CORRECTIONS = {
    ("MÂCON", "58"): ("71", "SAÔNE-ET-LOIRE"),
    ("CHALON-SUR-SAÔNE", "58"): ("71", "SAÔNE-ET-LOIRE"),
    ("CHALON-SUR-SAÔNE", "89"): ("71", "SAÔNE-ET-LOIRE"),
    ("METZ", "08"): ("57", "MOSELLE"),
    ("SARREGUEMINES", "08"): ("57", "MOSELLE"),
    ("SAVERNE", "51"): ("67", "BAS-RHIN"),
    ("NANCY", "08"): ("54", "MEURTHE-ET-MOSELLE"),
    ("MONTHERMÉ", "67"): ("08", "ARDENNES"),
}
for rec in records:
    key = (rec["commune"], rec["department_number"])
    if key in PORT_DEPT_CORRECTIONS:
        new_dept, new_dept_name = PORT_DEPT_CORRECTIONS[key]
        rec["department_number"] = new_dept
        rec["department_name"] = new_dept_name
        if new_dept in DEPT_REGION_MAP:
            rec["region"] = DEPT_REGION_MAP[new_dept]


# 7. Normalize curly apostrophe (U+2019) to straight apostrophe
for rec in records:
    for key in ("commune", "department_name", "region", "places"):
        rec[key] = rec[key].replace("\u2019", "'")

# 8. Merge records by commune + department
merged = {}
for rec in records:
    key = (rec["commune"], rec["department_number"])
    if key in merged:
        merged[key]["places"] = merged[key]["places"] + ", " + rec["places"]
        merged[key]["beach_flag"] = merged[key]["beach_flag"] or rec["beach_flag"]
        merged[key]["port_flag"] = merged[key]["port_flag"] or rec["port_flag"]
    else:
        merged[key] = dict(rec)

records = list(merged.values())

# 9. Remove records with empty places
records[:] = [r for r in records if r["places"].strip()]

# Re-write CSV
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    fields = [
        "year", "commune", "department_name", "department_number",
        "region", "beach_flag", "port_flag", "places",
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

print(f"Cleanup complete! {len(records)} records written to {OUTPUT_FILE}")


Cleanup complete! 167 records written to terragir_officiel_2025.csv
